# Evolución del precio de la tortilla en México (2007-2026)

**Visualización de datos, Maestría en Inteligencia Artificial, ITA**
Avance 3 del proyecto. Autora: Yomira del Carmen Rosales Martínez

**Pregunta analítica.** ¿Cómo ha evolucionado entre 2007 y 2026 el precio de la tortilla por entidad federativa y canal de venta (tortillerías y autoservicio)?

**Contenido del notebook**
1. Entorno y configuración
2. Carga de datos
3. Limpieza (con bitácora)
4. Datos procesados para otras herramientas (Tableau)
5. Exploración de datos (EDA)
6. Visualización estática
7. Visualización interactiva
8. Componente de Business Intelligence
9. Conclusiones

## 1. Entorno y configuración
Las rutas son **relativas** a la raíz del repositorio. Si se ejecuta el notebook desde la carpeta `notebooks/`, la primera línea del siguiente bloque cambia el directorio de trabajo.

In [ ]:
import os, platform
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RUTA_DATOS = Path("data/tortilla_prices.csv")
CARPETA_PROC = Path("data/processed")
CARPETA_FIG = Path("figuras")
YEAR_REF = 2025
CARPETA_PROC.mkdir(parents=True, exist_ok=True)
CARPETA_FIG.mkdir(exist_ok=True)

COLORES = {"Tortillerías": "#D55E00", "Autoservicio": "#0072B2"}
FUENTE = "Fuente: SNIIM, Secretaría de Economía (vía Kaggle, R. Chavelas). Precios nominales."

print("Python", platform.python_version(), "| pandas", pd.__version__,
      "| matplotlib", plt.matplotlib.__version__, "| seaborn", sns.__version__)

## 2. Carga de datos
- **Dataset:** *Tortilla Prices in Mexico*, Rick Chavelas (Kaggle). https://www.kaggle.com/datasets/richave/tortilla-prices-in-mexico
- **Fuente primaria:** SNIIM, Secretaría de Economía. http://www.economia-sniim.gob.mx/Tortilla.asp
- **Licencia:** CC0
- **Fecha de descarga:** 3 de Octubre de 2026
- El archivo es una copia derivada: el autor extrajo los datos del SNIIM mediante *web scraping* y los limpió sin documentar los pasos. Cada registro es el precio promedio por kilogramo (MXN, nominales) de un día, ciudad y canal.

In [ ]:
if not RUTA_DATOS.exists():
    raise FileNotFoundError(
        "No se encontró data/tortilla_prices.csv. Descárgalo de Kaggle "
        "y guárdalo en la carpeta data/.")

crudo = pd.read_csv(RUTA_DATOS)
print("Filas y columnas:", crudo.shape)
print("Rango de fechas:", pd.to_datetime(dict(year=crudo.Year, month=crudo.Month, day=crudo.Day)).agg(["min", "max"]).dt.date.tolist())
crudo.head()

## 3. Limpieza de datos
Decisiones tomadas (cada una queda registrada en la bitácora):
1. Eliminar espacios no separables (`\xa0`) en `State` y `City`.
2. Homologar nombres de entidades: "D.F." → "Ciudad de México" y "Edo. México" → "Estado de México".
3. Renombrar los canales con las etiquetas del SNIIM: *Mom and Pop Store* → **Tortillerías**; *Big Retail Store* → **Autoservicio**.
4. Construir una variable de fecha a partir de `Year`, `Month` y `Day`.
5. **No se imputan valores.** Los precios faltantes y el precio ≤ 0 se excluyen del cálculo.

In [ ]:
df = crudo.copy()
bitacora = [("Registros originales", len(df))]

n_nbsp = int(df["State"].str.contains("\xa0").sum() + df["City"].str.contains("\xa0").sum())
df["State"] = df["State"].str.replace("\xa0", " ", regex=False).str.strip()
df["City"] = df["City"].str.replace("\xa0", " ", regex=False).str.strip()
bitacora.append(("Celdas con espacios no separables corregidas", n_nbsp))

cambios = {"D.F.": "Ciudad de México", "Edo. México": "Estado de México"}
bitacora.append(("Registros con nombre de entidad homologado", int(df["State"].isin(cambios).sum())))
df["State"] = df["State"].replace(cambios)

df["Canal"] = df["Store type"].map({"Mom and Pop Store": "Tortillerías",
                                    "Big Retail Store": "Autoservicio"})
assert df["Canal"].notna().all()

df["Fecha"] = pd.to_datetime(dict(year=df["Year"], month=df["Month"], day=df["Day"]))

df["Precio"] = df["Price per kilogram"]
bitacora.append(("Precios faltantes (excluidos)", int(df["Precio"].isna().sum())))
bitacora.append(("Precios menores o iguales a 0 (excluidos)", int((df["Precio"] <= 0).sum())))

d = df[df["Precio"] > 0].copy()
bitacora.append(("Registros válidos para el análisis", len(d)))

assert d["State"].nunique() == 32
pd.DataFrame(bitacora, columns=["Paso", "Registros"])

In [ ]:
# Dónde se concentran los faltantes (Documentado en el reporte)
faltantes_por_year = df[df["Precio"].isna()].groupby("Year").size().rename("Precios faltantes")
faltantes_por_year.to_frame().T

## 4. Datos procesados para otras herramientas
Se exporta una tabla **mensual por entidad y canal** que usarán la gráfica interactiva y Tableau. Guarda `Suma_precio` y `N_obs` (no solo el promedio) para que el promedio ponderado de cualquier selección coincida con el de este notebook: `SUM(Suma_precio) / SUM(N_obs)`.

In [ ]:
mensual = (d.assign(Fecha_mes=d["Fecha"].dt.to_period("M").dt.to_timestamp())
             .groupby(["Fecha_mes", "State", "Canal"])
             .agg(Suma_precio=("Precio", "sum"),
                  N_obs=("Precio", "size"),
                  Ciudades=("City", "nunique"))
             .reset_index()
             .rename(columns={"Fecha_mes": "Fecha"}))
mensual["Year"] = mensual["Fecha"].dt.year
mensual["Month"] = mensual["Fecha"].dt.month
mensual["Precio_promedio"] = (mensual["Suma_precio"] / mensual["N_obs"]).round(4)
mensual = mensual[["Fecha", "Year", "Month", "State", "Canal",
                   "Suma_precio", "N_obs", "Ciudades", "Precio_promedio"]]

mensual.to_csv(CARPETA_PROC / "tortilla_mensual.csv", index=False, encoding="utf-8-sig")
print(mensual.shape)
mensual.head()

In [ ]:
anual = mensual.groupby(["Year", "Canal"])[["Suma_precio", "N_obs"]].sum()
anual["Precio_promedio"] = anual["Suma_precio"] / anual["N_obs"]
p = anual["Precio_promedio"].unstack()

print(p.loc[[2007, 2019, 2025]].round(2))
print("Crecimiento 2019-2025 (%):", ((p.loc[2025] / p.loc[2019] - 1) * 100).round(1).to_dict())
print("Brecha entre canales ($/kg):", (p["Tortillerías"] - p["Autoservicio"]).loc[[2007, 2019, 2025]].round(2).to_dict())

## 5. Exploración de datos (EDA)
Las cinco visualizaciones del Avance 2. Las figuras se guardan en `figuras/`.

In [ ]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
d25 = d[d["Year"] == YEAR_REF]

def guardar(fig, nombre):
    fig.text(0.01, 0.005, FUENTE, fontsize=7, color="gray")
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    fig.savefig(CARPETA_FIG / nombre, dpi=150)
    plt.show()
    plt.close(fig)

def fig1_histograma():
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(d25["Precio"], bins=30, color="#7f7f7f", edgecolor="white")
    mediana = d25["Precio"].median()
    ax.axvline(mediana, color="black", ls="--", lw=1)
    ax.text(mediana, ax.get_ylim()[1] * 0.95, f" Mediana: ${mediana:.1f}", va="top")
    ax.set_title(f"Distribución del precio por kilo de tortilla, {YEAR_REF}")
    ax.set_xlabel("Precio por kg (pesos nominales)")
    ax.set_ylabel("Registros (ciudad-día-canal)")
    guardar(fig, f"fig1_histograma_{YEAR_REF}.png")

def fig2_barras_entidad():
    est = d25.groupby("State")["Precio"].mean().sort_values()
    n_ciudades = d25.groupby("State")["City"].nunique()
    colores = ["#7f7f7f"] * len(est)
    colores[0] = colores[-1] = "#0072B2"
    fig, ax = plt.subplots(figsize=(7, 8))
    ax.barh(est.index, est.values, color=colores)
    ax.axvline(d25["Precio"].mean(), color="black", ls="--", lw=1)
    for i, (estado, v) in enumerate(est.items()):
        n = n_ciudades[estado]
        ax.text(v + 0.2, i, f"${v:.1f}  ({n} ciudad{'es' if n > 1 else ''})", va="center", fontsize=7)
    ax.set_xlim(0, est.max() * 1.25)
    ax.set_title(f"Precio promedio por kilo por entidad, {YEAR_REF} (ambos canales)")
    ax.set_xlabel("Precio por kg (pesos nominales)")
    guardar(fig, f"fig2_barras_entidad_{YEAR_REF}.png")

def fig3_mapa_calor():
    h = d[d["Year"] <= YEAR_REF].groupby(["State", "Year", "Canal"])["Precio"].mean().reset_index()
    orden = h[h["Year"] == YEAR_REF].groupby("State")["Precio"].mean().sort_values(ascending=False).index
    fig, axes = plt.subplots(1, 2, figsize=(13, 8), sharey=True)
    for ax, canal in zip(axes, ["Tortillerías", "Autoservicio"]):
        m = h[h["Canal"] == canal].pivot(index="State", columns="Year", values="Precio").reindex(orden)
        sns.heatmap(m, ax=ax, cmap="YlGnBu", vmin=h["Precio"].min(), vmax=h["Precio"].max(),
                    cbar=(canal == "Autoservicio"),
                    cbar_kws={"label": "Precio por kg (pesos nominales)"},
                    linewidths=0.3, linecolor="white")
        ax.set_title(canal); ax.set_xlabel("Año"); ax.set_ylabel("")
    fig.suptitle(f"Precio promedio por kilo por entidad, año y canal, 2007-{YEAR_REF}")
    guardar(fig, "fig3_mapa_calor.png")

def fig4_serie_tiempo():
    t = d.set_index("Fecha").groupby("Canal")["Precio"].resample("MS").mean().reset_index()
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for canal, g in t.groupby("Canal"):
        ax.plot(g["Fecha"], g["Precio"], color=COLORES[canal], lw=2, label=canal)
    ax.axvspan(pd.Timestamp("2020-01-01"), pd.Timestamp("2020-12-31"), color="gray", alpha=0.15)
    ax.set_ylim(0, None)
    ax.set_title("Precio promedio mensual por kilo de tortilla según canal, 2007-2026")
    ax.set_ylabel("Precio por kg (pesos nominales)")
    ax.legend(frameon=False)
    guardar(fig, "fig4_serie_tiempo.png")

def fig5_boxplot():
    b = d[d["Year"].isin([2007, 2013, 2019, 2025])]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.boxplot(data=b, x="Year", y="Precio", hue="Canal", hue_order=list(COLORES),
                palette=COLORES, fliersize=1.5, linewidth=0.9, ax=ax)
    ax.set_ylim(0, None)
    ax.set_title("Distribución del precio por kilo según canal, años seleccionados")
    ax.set_xlabel("Año"); ax.set_ylabel("Precio por kg (pesos nominales)")
    ax.legend(title="", frameon=False)
    guardar(fig, "fig5_boxplot.png")

for fig in (fig1_histograma, fig2_barras_entidad, fig3_mapa_calor, fig4_serie_tiempo, fig5_boxplot):
    fig()

## 6. Visualización estática


## 7. Visualización interactiva


## 8. Componente de Business Intelligence


## 9. Conclusiones